# Import libs

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import seaborn as sns
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, LeakyReLU, BatchNormalization
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from statsmodels.tsa.seasonal import seasonal_decompose
from joblib import Parallel, delayed

# Define the problem and collect data

In [ ]:
# from google.colab import drive
#drive.mount('/content/drive')

In [ ]:
df = pd.read_csv('/kaggle/input/pm2-5-dataset/Air Quality Ho Chi Minh City.csv')
df.head()

In [ ]:
data = df[["TSP", "PM2.5", 'O3', 'CO', 'NO2', 'SO2', 'Temperature', 'Humidity']]

In [ ]:
data.info()

In [ ]:
data.describe()

# Preprocess data

## Handle missing data

In [ ]:
data.isnull().sum()

### Fill missing values with linear interpolation

In [ ]:
data = data.interpolate(method='linear')
# If NaNs remain, fill with the nearest value
data = data.fillna(method='ffill').fillna(method='bfill')

In [ ]:
data.isnull().sum()

## Standardize timestamps

In [ ]:
# Convert the date column to datetime
df['date'] = pd.to_datetime(df['date'], errors='coerce')

# Identify NaT positions in the date column
missing_dates = df['date'].isna()

# Create a copy for interpolation
df_filled = df.copy()

# Interpolate date values
df_filled['date'] = df_filled['date'].interpolate(method='linear')
print(df_filled['date'].isna().sum())

In [ ]:
data['date'] = df_filled['date']
data.set_index('date', inplace=True)
data.head()

## Handle outliers

In [ ]:
# Plot box plot before processing
plt.figure(figsize=(12, 6))
sns.boxplot(data=data)
plt.title('Box Plot of Numeric Features (Before Outlier Handling)')
plt.xticks(rotation=45)
plt.show()

In [ ]:
def detect_and_interpolate_outliers(data, column):
    """
    Detect outliers using the IQR method and replace them with interpolated values.

    Parameters:
    - data: DataFrame containing the data
    - column: Column name to inspect and process for outliers

    Returns:
    - DataFrame with outliers replaced by interpolated values
    """
    Q1 = data[column].quantile(0.25)  # First quartile (25%)
    Q3 = data[column].quantile(0.75)  # Third quartile (75%)
    IQR = Q3 - Q1                     # Interquartile range

    lower_bound = Q1 - 1.5 * IQR       # Lower bound
    upper_bound = Q3 + 1.5 * IQR       # Upper bound

    # Identify outlier positions
    outliers_mask = (data[column] < lower_bound) | (data[column] > upper_bound)

    # Replace outliers with NaN for interpolation
    data.loc[outliers_mask, column] = np.nan

    # Use linear interpolation to fill missing values
    data[column] = data[column].interpolate(method='linear')

    return data

In [ ]:
for column in data.columns:
    data = detect_and_interpolate_outliers(data, column)

In [ ]:
# Plot box plot after processing
plt.figure(figsize=(12, 6))
sns.boxplot(data=data)
plt.title('Box Plot of Numeric Features (After Outlier Handling)')
plt.xticks(rotation=45)
plt.show()

# Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(12,5))
sns.distplot(data['PM2.5'],bins=50)
plt.title('Distribution of the hourly recorded PM2.5 concetration in the air\nin Aotizhongxin-Beijin',
          fontsize=16)
plt.show()

In [ ]:
daily_data = data[['PM2.5']].copy() # Create a copy of the PM2.5 column
daily_data['date'] = data.index
daily_data = daily_data.set_index('date') # Set 'date' as the index
daily_data = daily_data.resample('D').median()
# Fill NaN values after resampling with backfill
daily_data = daily_data.bfill()
decomposition = seasonal_decompose(daily_data, model='additive') # Correct model parameter to 'additive'

# plot the data
with plt.style.context('fivethirtyeight'):
    decomposition.trend.plot(figsize=(12,5),style='k-',linewidth=.9,legend=False)
    plt.xlabel('Date',fontsize=14)
    plt.ylabel('PM2.5 concentration (ug/m^3)',fontsize=14)
    plt.title('Daily trend in the hourly recorded PM2.5 concentration in\nthe air in Ho Chi Minh',fontsize=16)
    plt.grid(axis='x')
    plt.tight_layout()
    plt.show()

In [ ]:
# Extract the month from the datetime index and create a new 'month' column
data['month'] = data.index.month

# Now you can proceed with your original code
monthly_data = data[['month','PM2.5']]
months = ['January','February','March','April','May','June','July',
         'August','September','October','November','December']
ordered_monthdf = pd.DataFrame(months,columns=['month'])
map_dict = {}
for i,j in enumerate(months):
    map_dict.setdefault(i+1,j)
monthly_data.loc[:, 'month'] = monthly_data['month'].map(map_dict)
monthly_average = monthly_data.groupby('month').median()
monthly_average = pd.merge(ordered_monthdf,monthly_average,left_on='month',right_index=True)
monthly_average = np.round(monthly_average,1)
monthly_average = monthly_average.set_index('month')

# plot the data
with plt.style.context('ggplot'):
    monthly_average.plot(figsize=(12,5),legend=False,kind='bar',linewidth=.9)
    plt.xlabel('Month',fontsize=14)
    plt.ylabel('PM2.5 concentration (ug/m^3)',fontsize=14)
    plt.title('Monthly average of the hourly recorded PM2.5 concentration in\nthe air in Ho Chi Minh',fontsize=16)
    plt.grid(axis='x')
    plt.tight_layout()
    plt.show()

In [ ]:
data['hour'] = data.index.hour  # Extract hour from DateTime index

hourly_data = data[['hour', 'PM2.5']]
hrs = ['12 AM','1 AM','2 AM','3 AM','4 AM','5 AM','6 AM','7 AM','8 AM','9 AM','10 AM',
      '11 AM','12 PM','1 PM','2 PM','3 PM','4 PM','5 PM','6 PM','7 PM',
      '8 PM','9 PM','10 PM','11 PM']

hour_dict = {i: j for i, j in enumerate(hrs)}

hourly_data = hourly_data.groupby('hour').median().reset_index()
hourly_data['hour'] = hourly_data['hour'].map(hour_dict)
hourly_data = hourly_data.set_index('hour')

# Plot
with plt.style.context('ggplot'):
    hourly_data.plot(figsize=(12,8), legend=False, kind='barh', linewidth=.9)
    plt.ylabel('Hours', fontsize=14)
    plt.xlabel('PM2.5 concentration (ug/m^3)', fontsize=14)
    plt.title('Average recorded PM2.5 concentration in the air in Ho Chi Minh\nby the hour of the day', fontsize=16)
    plt.grid(axis='y')
    plt.tight_layout()
    plt.show()


In [ ]:
plt.figure(figsize=(13,9))
correlation_data = data
sns.heatmap(correlation_data.corr(),cmap=plt.cm.Reds,annot=True)
plt.title('Heatmap displaying the correlation matrix of the variables',fontsize=16)
plt.show()

## Normalize data

In [ ]:
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(data)

In [ ]:
# Create time-series sequences for LSTM
def create_sequences(data, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i + seq_length])
        y.append(data[i + seq_length, 0])  # 0 is the PM2.5 column
    return np.array(X), np.array(y)
    
seq_length = 24  # Predict from the previous 24 hours
PM2_5= ['PM2.5']
data = df[PM2_5].values.astype(np.float32)  # Cast to float32
X, y = create_sequences(data, seq_length)

print(f"Shape of X: {X.shape}")  # (samples, 24 hours, features)
print(f"Shape of y: {y.shape}")

In [ ]:
# Split into train (70%) and test (30%)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, shuffle=False)

# Split the remaining data into test (15%) and validation (15%)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, shuffle=False)

In [ ]:
# Build the LSTM model
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(seq_length, 1), kernel_regularizer=l2(0.001)),
    LeakyReLU(alpha=0.1),
    BatchNormalization(),
    Dropout(0.2),

    LSTM(32, return_sequences=False, kernel_regularizer=l2(0.001)),
    LeakyReLU(alpha=0.1),
    BatchNormalization(),
    Dropout(0.2),

    Dense(1, kernel_regularizer=l2(0.001))  # Predict PM2.5 value
])

# Use Adam for optimization
model.compile(loss="mse", optimizer=Adam(), metrics=["mse"])

# Display the model
model.summary()

In [ ]:
# Train the model
history = model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_val, y_val), verbose=0)

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training & Validation Loss')
plt.legend()
plt.show()

## Predict on the test set

In [ ]:
# Predict on the test set
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Compute evaluation metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

# Print results
print(f"MAE: {mae:.4f}")
print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R2 Score: {r2:.4f}")

In [ ]:
# Plot actual vs predicted values
plt.figure(figsize=(10, 5))
plt.plot(y_test[:100], label="Actual PM2.5", marker='o', linestyle='dashed')
plt.plot(y_pred[:100], label="Predicted PM2.5", marker='x', linestyle='dashed')
plt.xlabel("Time (samples)")
plt.ylabel("PM2.5 Value")
plt.title("Actual vs Predicted PM2.5")
plt.legend()
plt.show()

In [ ]:
import numpy as np

def predict_future(model, test_data, seq_length=24, n_days=3):
    """
    Predict PM2.5 for the next n days based on the test set.

    Args:
        model: Trained LSTM model.
        test_data: Test dataset (numpy array).
        seq_length: Length of each input sequence (default: 24 hours).
        n_days: Number of days to predict (24 hourly values per day).

    Returns:
        predictions: List of predicted PM2.5 values for the next n days.
    """
    future_predictions = []
    input_seq = test_data[-1].reshape(1, seq_length, -1)  # Use the last test sample as input

    for _ in range(n_days * 24):  # Predict each hour for the next n days
        pred_pm25 = model.predict(input_seq, verbose=0)[0][0]  # Predict PM2.5
        future_predictions.append(pred_pm25)

        # Update input_seq to continue forecasting
        new_input = np.roll(input_seq, shift=-1, axis=1)  # Shift the sequence data
        new_input[0, -1, 0] = pred_pm25  # Assign the new value to the last position
        input_seq = new_input

    return future_predictions

# Predict the next 3 days (72 hours)
future_pm25 = predict_future(model, X_test, n_days=3)

# Print daily results
for i in range(3):
    daily_values = future_pm25[i * 24:(i + 1) * 24]
    print(f"Day {i+1}: Average PM2.5 = {np.mean(daily_values):.4f}")
